# Notebook 3: TextCNN

**Research question:** Can a spam classifier trained on SMS messages generalize to emails?

This notebook trains a compact TextCNN-style classifier from scratch, inspired by [Kim (2014)](https://aclanthology.org/D14-1181/). It adds validation-only hyperparameter selection and repeated training seeds to measure whether the cross-domain result is stable.

## Experiment design

- Reuse the frozen, leakage-free splits from Notebook 1.
- Keep the vocabulary at 30,000 tokens and the input length at 256 tokens.
- Compare four predeclared TextCNN configurations, each changing at most one choice from the reference model.
- Tune SMS and Enron separately using only their own training and validation splits.
- Rank candidates by mean validation macro-F1 across seeds `13`, `42`, and `73`.
- Lock one configuration per training source before accessing any test split.
- Retrain each locked configuration with seeds `13`, `42`, `73`, `101`, and `137`, then evaluate all four domain pairs.
- Report every seed and mean ± sample standard deviation; seed `42` is singled out only for representative diagnostic plots, never as the best run.

Early stopping restores the checkpoint with the lowest validation loss, while hyperparameter selection compares completed runs by validation macro-F1. The threshold remains fixed at `0.5`. Sequence length is not tuned here because it is reserved for a separate controlled length experiment. Seeds `13`, `42`, and `73` are reused for retraining after configuration lock; `101` and `137` are additional reporting seeds, so seed variation is interpreted descriptively rather than as a fully independent HPO replication. The test sets were inspected in the initial project, so the new scores are treated as fixed confirmation benchmarks rather than pristine unseen holdouts.

In [ ]:
from pathlib import Path
import gc
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kbozukov-coke/cross-domain-spam-detection.git'
PROJECT_NAME = 'cross-domain-spam-detection'
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working') / PROJECT_NAME
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'],
            check=True,
        )
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay

from src.data import load_prepared_splits, summarize_splits
from src.evaluation import (
    aggregate_seed_metrics,
    binary_classification_metrics,
    build_prediction_table,
)
from src.modeling import run_transfer_experiments
from src.protocol import (
    CONFIRMATION_TRAINING_SEEDS,
    DATA_SPLIT_SEED,
    DECISION_THRESHOLD,
    FINAL_TRAINING_SEEDS,
    REFERENCE_TRAINING_SEED,
    SELECTION_METRIC,
)
from src.textcnn import evaluate_textcnn, fit_textcnn

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)

print(f'Running in: {"Kaggle" if IS_KAGGLE else "local environment"}')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')

## Load the prepared data

The preprocessing and partitions are identical to the previous notebooks. The split seed stays fixed; training seeds never recreate the data partitions.

In [ ]:
splits, _ = load_prepared_splits(random_state=DATA_SPLIT_SEED)
split_summary = summarize_splits(splits)
split_summary.loc[:, ['dataset', 'split', 'rows', 'ham', 'spam', 'spam_rate']]

## Predeclared TextCNN search space

The search is intentionally small and interpretable. Relative to the reference model, the alternatives change only the convolution width, dropout, or learning rate. All capacity settings and the 256-token input length remain fixed.

In [ ]:
TEXTCNN_BASE_CONFIG = {
    'max_tokens': 30_000,
    'sequence_length': 256,
    'embedding_dim': 128,
    'filters': 128,
    'dense_units': 64,
    'epochs': 8,
    'batch_size': 64,
    'patience': 2,
}

TEXTCNN_CANDIDATES = pd.DataFrame([
    {
        'candidate_order': 0,
        'config_id': 'reference',
        'kernel_size': 5,
        'dropout': 0.4,
        'learning_rate': 1e-3,
    },
    {
        'candidate_order': 1,
        'config_id': 'kernel_3',
        'kernel_size': 3,
        'dropout': 0.4,
        'learning_rate': 1e-3,
    },
    {
        'candidate_order': 2,
        'config_id': 'dropout_0_5',
        'kernel_size': 5,
        'dropout': 0.5,
        'learning_rate': 1e-3,
    },
    {
        'candidate_order': 3,
        'config_id': 'lr_5e_4',
        'kernel_size': 5,
        'dropout': 0.4,
        'learning_rate': 5e-4,
    },
])

TUNING_SEEDS = tuple(CONFIRMATION_TRAINING_SEEDS)
REPORTING_SEEDS = tuple(FINAL_TRAINING_SEEDS)
assert SELECTION_METRIC == 'macro_f1'

def fit_config_for(candidate):
    return {
        **TEXTCNN_BASE_CONFIG,
        'kernel_size': int(candidate.kernel_size),
        'dropout': float(candidate.dropout),
        'learning_rate': float(candidate.learning_rate),
    }

display(TEXTCNN_CANDIDATES)
print(f'Validation selection metric: {SELECTION_METRIC}')
print(f'Tuning seeds: {TUNING_SEEDS}')
print(f'Final reporting seeds: {REPORTING_SEEDS}')
print(
    'Planned sequential fits: '
    f'{2 * len(TEXTCNN_CANDIDATES) * len(TUNING_SEEDS)} tuning + '
    f'{2 * len(REPORTING_SEEDS)} locked test runs'
)

## Validation-only tuning

Every candidate is trained with the three confirmation seeds. It is scored only on the validation split of the same source domain: SMS candidates on SMS validation and Enron candidates on Enron validation. No target-domain validation data and no test data are accessed in this phase.

In [ ]:
tuning_rows = []

for train_domain in ('sms', 'enron'):
    for candidate in TEXTCNN_CANDIDATES.itertuples(index=False):
        candidate_config = fit_config_for(candidate)

        for training_seed in TUNING_SEEDS:
            print(
                f'Tuning {train_domain.upper()} | '
                f'{candidate.config_id} | seed {training_seed}'
            )
            model, history, training_info = fit_textcnn(
                splits[train_domain]['train'],
                splits[train_domain]['validation'],
                random_state=training_seed,
                verbose=0,
                **candidate_config,
            )
            validation_result = evaluate_textcnn(
                model,
                splits[train_domain]['validation'],
                train_domain=train_domain,
                test_domain=train_domain,
                training_info=training_info,
            )
            tuning_rows.append({
                'train_domain': train_domain,
                'config_id': candidate.config_id,
                'candidate_order': candidate.candidate_order,
                'training_seed': training_seed,
                'kernel_size': candidate.kernel_size,
                'dropout': candidate.dropout,
                'learning_rate': candidate.learning_rate,
                'validation_macro_f1': validation_result['macro_f1'],
                'validation_spam_f1': validation_result['f1'],
                'validation_balanced_accuracy': validation_result['balanced_accuracy'],
                'validation_mcc': validation_result['mcc'],
                'validation_pr_auc': validation_result['pr_auc'],
                'best_validation_loss': min(history.history['val_loss']),
                'epochs_trained': training_info['epochs_trained'],
                'training_seconds': training_info['training_seconds'],
            })

            del model, history
            tf.keras.backend.clear_session()
            gc.collect()

tuning_results = pd.DataFrame(tuning_rows)
expected_tuning_rows = (
    2 * len(TEXTCNN_CANDIDATES) * len(TUNING_SEEDS)
)
assert len(tuning_results) == expected_tuning_rows
assert not tuning_results.duplicated(
    ['train_domain', 'config_id', 'training_seed']
).any()

print(f'Finished {len(tuning_results)} validation-only tuning runs.')

## Select and lock one configuration per source

Candidates are ranked by mean validation macro-F1. A lower macro-F1 standard deviation breaks an exact mean tie, followed by the predeclared candidate order. This deterministic rule is fixed before test evaluation.

In [ ]:
tuning_summary = (
    tuning_results
    .groupby(
        [
            'train_domain',
            'config_id',
            'candidate_order',
            'kernel_size',
            'dropout',
            'learning_rate',
        ],
        as_index=False,
    )
    .agg(
        validation_macro_f1_mean=('validation_macro_f1', 'mean'),
        validation_macro_f1_std=('validation_macro_f1', 'std'),
        validation_spam_f1_mean=('validation_spam_f1', 'mean'),
        validation_balanced_accuracy_mean=(
            'validation_balanced_accuracy', 'mean'
        ),
        validation_mcc_mean=('validation_mcc', 'mean'),
        validation_pr_auc_mean=('validation_pr_auc', 'mean'),
        best_validation_loss_mean=('best_validation_loss', 'mean'),
        epochs_trained_mean=('epochs_trained', 'mean'),
        training_seconds_mean=('training_seconds', 'mean'),
        n_seeds=('training_seed', 'nunique'),
    )
)

ranked_tuning = tuning_summary.sort_values(
    [
        'train_domain',
        'validation_macro_f1_mean',
        'validation_macro_f1_std',
        'candidate_order',
    ],
    ascending=[True, False, True, True],
    kind='stable',
)
selected_rows = (
    ranked_tuning.groupby('train_domain', sort=False).head(1).copy()
)
selected_ids = selected_rows.set_index('train_domain')['config_id'].to_dict()
tuning_summary['selected'] = tuning_summary.apply(
    lambda row: selected_ids[row['train_domain']] == row['config_id'],
    axis=1,
)

selected_configs = {}
for domain, config_id in selected_ids.items():
    candidate = TEXTCNN_CANDIDATES.query(
        'config_id == @config_id'
    ).iloc[0]
    selected_configs[domain] = {
        **TEXTCNN_BASE_CONFIG,
        'kernel_size': int(candidate['kernel_size']),
        'dropout': float(candidate['dropout']),
        'learning_rate': float(candidate['learning_rate']),
    }

assert set(selected_configs) == {'sms', 'enron'}
assert tuning_summary['n_seeds'].eq(len(TUNING_SEEDS)).all()

display_columns = [
    'train_domain', 'config_id', 'kernel_size', 'dropout',
    'learning_rate', 'validation_macro_f1_mean',
    'validation_macro_f1_std', 'validation_spam_f1_mean',
    'best_validation_loss_mean', 'n_seeds', 'selected',
]
display_tuning = tuning_summary.loc[:, display_columns].copy()
score_columns = [
    'validation_macro_f1_mean', 'validation_macro_f1_std',
    'validation_spam_f1_mean', 'best_validation_loss_mean',
]
display_tuning[score_columns] = display_tuning[score_columns].round(4)
display(
    display_tuning.sort_values(
        ['train_domain', 'candidate_order']
        if 'candidate_order' in display_tuning.columns
        else ['train_domain', 'config_id']
    )
)
selected_rows.loc[:, display_columns[:-1]].round(4)

## Locked five-seed confirmation experiment

The two selected configurations are now frozen. Each is retrained from scratch with all five reporting seeds and evaluated on both fixed test domains. Models are processed sequentially and released immediately; only small metric tables, seed-42 histories, and seed-42 prediction tables remain in memory.

In [ ]:
textcnn_seed_rows = []
representative_histories = {}
representative_predictions = {}

for train_domain in ('sms', 'enron'):
    selected_config_id = selected_ids[train_domain]
    selected_config = selected_configs[train_domain]

    for training_seed in REPORTING_SEEDS:
        print(
            f'Final {train_domain.upper()} model | '
            f'{selected_config_id} | seed {training_seed}'
        )
        model, history, training_info = fit_textcnn(
            splits[train_domain]['train'],
            splits[train_domain]['validation'],
            random_state=training_seed,
            verbose=1,
            **selected_config,
        )

        if training_seed == REFERENCE_TRAINING_SEED:
            representative_histories[train_domain] = pd.DataFrame(
                history.history
            )

        for test_domain in ('sms', 'enron'):
            test_frame = splits[test_domain]['test']
            test_texts = (
                test_frame['text'].astype(str).to_numpy(dtype=object)
            )
            spam_probabilities = model.predict(
                test_texts, verbose=0
            ).reshape(-1)
            metrics = binary_classification_metrics(
                test_frame['label'],
                spam_probabilities,
                threshold=DECISION_THRESHOLD,
            )
            textcnn_seed_rows.append({
                'model': 'TextCNN',
                'config_id': selected_config_id,
                'training_seed': training_seed,
                'train_domain': train_domain,
                'test_domain': test_domain,
                'setting': (
                    'in-domain'
                    if train_domain == test_domain
                    else 'cross-domain'
                ),
                'test_rows': len(test_frame),
                **metrics,
                **training_info,
                'kernel_size': selected_config['kernel_size'],
                'dropout': selected_config['dropout'],
                'learning_rate': selected_config['learning_rate'],
                'sequence_length': selected_config['sequence_length'],
            })

            if training_seed == REFERENCE_TRAINING_SEED:
                representative_predictions[(train_domain, test_domain)] = (
                    build_prediction_table(
                        test_frame,
                        spam_probabilities,
                        model='TextCNN',
                        training_seed=training_seed,
                        train_domain=train_domain,
                        test_domain=test_domain,
                        threshold=DECISION_THRESHOLD,
                    )
                )

        del model, history
        tf.keras.backend.clear_session()
        gc.collect()

textcnn_seed_results = pd.DataFrame(textcnn_seed_rows)
expected_test_rows = 4 * len(REPORTING_SEEDS)
assert len(textcnn_seed_results) == expected_test_rows
assert not textcnn_seed_results.duplicated(
    ['training_seed', 'train_domain', 'test_domain']
).any()
assert set(representative_histories) == {'sms', 'enron'}
assert len(representative_predictions) == 4

print(f'Finished {len(textcnn_seed_results)} locked test evaluations.')

## Representative learning curves

The curves below show the predeclared reference seed `42` only. They are useful for diagnosing convergence, but they are not substituted for the five-seed result table.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for axis, domain in zip(axes, ('sms', 'enron')):
    history = representative_histories[domain]
    epochs = np.arange(1, len(history) + 1)
    axis.plot(epochs, history['loss'], marker='o', label='train')
    axis.plot(epochs, history['val_loss'], marker='o', label='validation')
    axis.set_title(f'{domain.upper()} loss (seed 42)')
    axis.set_xlabel('Epoch')
    axis.set_ylabel('Binary cross-entropy')
    axis.legend()

plt.tight_layout()
plt.show()

## Results for every seed

No best seed is selected. Precision, recall, and F1 use spam as the positive class. The additional metrics expose class imbalance and ranking performance instead of relying on accuracy alone.

In [ ]:
individual_metric_columns = [
    'accuracy', 'precision', 'recall', 'f1', 'macro_f1',
    'balanced_accuracy', 'mcc', 'roc_auc', 'pr_auc',
]
individual_display = textcnn_seed_results.loc[:, [
    'training_seed', 'train_domain', 'test_domain', 'setting',
    'config_id', 'ham_support', 'spam_support',
    *individual_metric_columns,
]].copy()
individual_display[individual_metric_columns] = (
    individual_display[individual_metric_columns].round(3)
)
individual_display.sort_values(
    ['train_domain', 'test_domain', 'training_seed']
)

## Training-seed variability

Mean and sample standard deviation summarize variation from initialization and training order. They do not quantify finite-test-sample uncertainty; Notebook 2 reports stratified bootstrap intervals for that separate source of uncertainty.

In [ ]:
_, textcnn_seed_summary = aggregate_seed_metrics(
    textcnn_seed_results,
    group_columns=[
        'model', 'config_id', 'train_domain', 'test_domain', 'setting'
    ],
    metric_columns=individual_metric_columns,
)

summary_columns = [
    'model', 'config_id', 'train_domain', 'test_domain', 'setting',
    'f1_mean', 'f1_std', 'macro_f1_mean', 'macro_f1_std',
    'precision_mean', 'precision_std', 'recall_mean', 'recall_std',
    'balanced_accuracy_mean', 'balanced_accuracy_std',
    'mcc_mean', 'mcc_std', 'roc_auc_mean', 'roc_auc_std',
    'pr_auc_mean', 'pr_auc_std', 'f1_count',
]
summary_display = textcnn_seed_summary.loc[:, summary_columns].copy()
summary_score_columns = [
    column for column in summary_display.columns
    if column.endswith('_mean') or column.endswith('_std')
]
summary_display[summary_score_columns] = (
    summary_display[summary_score_columns].round(3)
)
summary_display

## Compare the five-seed TextCNN result with the deterministic baseline

The TF-IDF baseline is refitted once on the same frozen splits. Its point estimate is compared with the TextCNN mean and standard deviation; no artificial training-seed variance is assigned to the deterministic model.

In [ ]:
_, baseline_results = run_transfer_experiments(
    splits,
    random_state=REFERENCE_TRAINING_SEED,
)

baseline_reference = baseline_results.loc[
    :, ['train_domain', 'test_domain', 'f1']
].rename(columns={'f1': 'baseline_f1'})

comparison_summary = (
    textcnn_seed_summary.merge(
        baseline_reference,
        on=['train_domain', 'test_domain'],
        how='left',
        validate='one_to_one',
    )
)
comparison_summary['mean_change_vs_baseline'] = (
    comparison_summary['f1_mean'] - comparison_summary['baseline_f1']
)
comparison_summary.loc[:, [
    'train_domain', 'test_domain', 'config_id',
    'baseline_f1', 'f1_mean', 'f1_std',
    'mean_change_vs_baseline', 'f1_count',
]].round(3)

In [ ]:
PAIR_ORDER = [
    'SMS -> SMS',
    'SMS -> ENRON',
    'ENRON -> SMS',
    'ENRON -> ENRON',
]

plot_results = textcnn_seed_results.copy()
plot_results['experiment'] = (
    plot_results['train_domain'].str.upper()
    + ' -> '
    + plot_results['test_domain'].str.upper()
)
baseline_plot = baseline_results.copy()
baseline_plot['experiment'] = (
    baseline_plot['train_domain'].str.upper()
    + ' -> '
    + baseline_plot['test_domain'].str.upper()
)

fig, axis = plt.subplots(figsize=(10, 5))
sns.stripplot(
    data=plot_results,
    x='experiment',
    y='f1',
    order=PAIR_ORDER,
    color='tab:blue',
    jitter=0.12,
    size=7,
    alpha=0.75,
    ax=axis,
)

for position, experiment in enumerate(PAIR_ORDER):
    values = plot_results.query(
        'experiment == @experiment'
    )['f1']
    axis.errorbar(
        position,
        values.mean(),
        yerr=values.std(ddof=1),
        fmt='o',
        color='black',
        capsize=5,
        label='TextCNN mean ± SD' if position == 0 else None,
    )
    baseline_value = baseline_plot.query(
        'experiment == @experiment'
    )['f1'].iloc[0]
    axis.scatter(
        position,
        baseline_value,
        marker='s',
        s=65,
        color='tab:orange',
        label='TF-IDF baseline' if position == 0 else None,
        zorder=4,
    )

axis.set_title('TextCNN seed variability and deterministic baseline')
axis.set_xlabel('Train -> test domain')
axis.set_ylabel('Spam F1')
axis.set_ylim(0, 1)
axis.legend()
plt.tight_layout()
plt.show()

## Primary transfer stability

For each matching seed, the SMS in-domain F1 is paired with the SMS-to-Enron F1. This produces a seed-level transfer gap. The change relative to the deterministic baseline is also shown for every seed instead of only for the most favorable run.

In [ ]:
sms_seed_f1 = (
    textcnn_seed_results.query("train_domain == 'sms'")
    .pivot(index='training_seed', columns='test_domain', values='f1')
    .rename(columns={'sms': 'sms_to_sms_f1', 'enron': 'sms_to_enron_f1'})
    .reset_index()
)
sms_seed_f1['transfer_gap'] = (
    sms_seed_f1['sms_to_sms_f1'] - sms_seed_f1['sms_to_enron_f1']
)

baseline_sms_to_enron = baseline_results.query(
    "train_domain == 'sms' and test_domain == 'enron'"
)['f1'].iloc[0]
sms_seed_f1['change_vs_baseline'] = (
    sms_seed_f1['sms_to_enron_f1'] - baseline_sms_to_enron
)

transfer_stability_summary = pd.DataFrame({
    'quantity': [
        'SMS -> Enron F1',
        'SMS in-domain -> Enron transfer gap',
        'SMS -> Enron change vs baseline',
    ],
    'mean': [
        sms_seed_f1['sms_to_enron_f1'].mean(),
        sms_seed_f1['transfer_gap'].mean(),
        sms_seed_f1['change_vs_baseline'].mean(),
    ],
    'sample_std': [
        sms_seed_f1['sms_to_enron_f1'].std(ddof=1),
        sms_seed_f1['transfer_gap'].std(ddof=1),
        sms_seed_f1['change_vs_baseline'].std(ddof=1),
    ],
    'n_seeds': len(sms_seed_f1),
})

display(sms_seed_f1.round(3))
transfer_stability_summary.round(3)

## Representative confusion matrices

These matrices use seed `42` and the locked configuration for each training source. The aggregated tables above remain the basis for the conclusions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for axis, train_domain, test_domain in zip(
    axes.ravel(),
    ('sms', 'sms', 'enron', 'enron'),
    ('sms', 'enron', 'sms', 'enron'),
):
    predictions = representative_predictions[(train_domain, test_domain)]
    ConfusionMatrixDisplay.from_predictions(
        predictions['label'],
        predictions['prediction'],
        display_labels=['ham', 'spam'],
        colorbar=False,
        ax=axis,
    )
    axis.set_title(
        f'Train {train_domain.upper()} -> '
        f'Test {test_domain.upper()} (seed 42)'
    )

plt.tight_layout()
plt.show()

## Representative SMS -> Enron errors

The five highest-margin seed-42 errors provide a qualitative diagnostic. They are examples, not an additional selection criterion.

In [ ]:
sms_to_enron = representative_predictions[('sms', 'enron')].copy()
cross_domain_errors = sms_to_enron.loc[~sms_to_enron['correct']].copy()
cross_domain_errors['error_type'] = cross_domain_errors['label'].map(
    {0: 'false positive', 1: 'false negative'}
)
cross_domain_errors['confidence'] = (
    cross_domain_errors['spam_probability'] - DECISION_THRESHOLD
).abs()

display(
    cross_domain_errors['error_type']
    .value_counts()
    .rename('errors')
    .to_frame()
)
cross_domain_errors.sort_values(
    'confidence', ascending=False
).loc[
    :, ['error_type', 'spam_probability', 'text']
].head(5)

## TextCNN conclusion

The conclusion is generated from all five locked runs. A positive average change over the baseline is reported descriptively, together with its seed variability and the number of seeds that improve; it is not presented as a significance test.

In [ ]:
primary_summary = textcnn_seed_summary.query(
    "train_domain == 'sms' and test_domain == 'enron'"
).iloc[0]
gap_mean = sms_seed_f1['transfer_gap'].mean()
gap_std = sms_seed_f1['transfer_gap'].std(ddof=1)
change_mean = sms_seed_f1['change_vs_baseline'].mean()
change_std = sms_seed_f1['change_vs_baseline'].std(ddof=1)
seeds_above_baseline = int(
    (sms_seed_f1['change_vs_baseline'] > 0).sum()
)

print(
    'TextCNN SMS -> Enron spam F1: '
    f"{primary_summary['f1_mean']:.3f} ± "
    f"{primary_summary['f1_std']:.3f} SD "
    f"(n={int(primary_summary['f1_count'])})"
)
print(f'TF-IDF SMS -> Enron spam F1: {baseline_sms_to_enron:.3f}')
print(
    'Mean TextCNN change vs baseline: '
    f'{change_mean:+.3f} ± {change_std:.3f} SD'
)
print(
    'Mean TextCNN transfer gap: '
    f'{gap_mean:.3f} ± {gap_std:.3f} SD'
)
print(
    'Seeds above the deterministic baseline: '
    f'{seeds_above_baseline}/{len(REPORTING_SEEDS)}'
)

if change_mean > 0:
    print(
        'Conclusion: TextCNN improves SMS-to-email F1 on average, '
        'but the remaining transfer gap is substantial.'
    )
else:
    print(
        'Conclusion: TextCNN does not improve SMS-to-email F1 on average, '
        'and cross-domain generalization remains weak.'
    )

## Export reusable results

Four compact CSV files are written for the next notebooks: every tuning run, the tuning summary and selected configurations, every locked test run, and the five-seed metric summary. The legacy seed-42 `textcnn_results.csv` is deliberately not overwritten until Notebook 4 is migrated to the new multi-seed artifacts.

In [ ]:
SOURCE_NOTEBOOK = (
    'https://www.kaggle.com/code/kaloyanbozukov/'
    'notebook-3-textcnn-cross-domain-experiment'
)
results_directory = PROJECT_ROOT / 'results'
results_directory.mkdir(parents=True, exist_ok=True)

exports = {
    'textcnn_tuning_results.csv': tuning_results,
    'textcnn_tuning_summary.csv': tuning_summary,
    'textcnn_seed_results.csv': textcnn_seed_results,
    'textcnn_seed_summary.csv': textcnn_seed_summary,
}

for filename, frame in exports.items():
    output_path = results_directory / filename
    (
        frame.assign(source_notebook=SOURCE_NOTEBOOK)
        .to_csv(output_path, index=False)
    )
    print(f'Saved {len(frame):>2} rows to {output_path}')